In [ ]:
!pip install -q pandas numpy pillow tqdm matplotlib

# =========================
# Cell 1 - 配置信息与依赖安装
# =========================

import os
import io
import json
import math
import random
import shutil
import hashlib
from copy import deepcopy
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance, ImageFilter, ImageDraw
from tqdm.auto import tqdm

# 项目根目录。多机/迁移时推荐设置环境变量 FIRE_ROOT；否则默认 notebook 位于项目子目录时使用上一级目录。
ROOT = Path(os.environ.get("FIRE_ROOT", "..")).resolve()
print("ROOT =", ROOT)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
RANDOM_SEED = 35
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 原比赛数据：只从这里固定抽验证集；其余进入训练集；不设测试集。
FIRE_IMAGE_DIRS = [ROOT / "image_fire", ROOT / "images_fire"]
NO_FIRE_IMAGE_DIRS = [ROOT / "image_no_fire", ROOT / "images_no_fire"]

# 额外数据集：全部只进入 train，不进入 val/test。
EXTRA_DATASET_DIR = ROOT / "extra_datasets"

# 输出目录。
SPLIT_DIR = ROOT / "fire_splits"
AUG_IMAGE_DIR = SPLIT_DIR / "augmented_images_full"
AUG_LABELME_DIR = SPLIT_DIR / "augmented_labelme_jsons_full"
MASK_NPZ_CACHE_DIR = SPLIT_DIR / "mask_npz_cache"
DEGRADED_IMAGE_CACHE_DIR = SPLIT_DIR / "degraded_image_cache"

DATASET_REGISTRY_CSV = SPLIT_DIR / "dataset_registry.csv"
SPLIT_MANIFEST_CSV = SPLIT_DIR / "split_manifest.csv"
TRAIN_REAL_CSV = SPLIT_DIR / "train_all_real_only.csv"
TRAIN_ALL_CSV = SPLIT_DIR / "train_all.csv"
VAL_CSV = SPLIT_DIR / "val_all_datasets_real_only.csv"
TEST_CSV = SPLIT_DIR / "test_all_datasets_real_only.csv"  # 保留空表，方便兼容旧工具。
MASK_CACHE_INDEX_CSV = SPLIT_DIR / "mask_cache_index.csv"
DEGRADED_INDEX_CSV = SPLIT_DIR / "degraded_image_cache_index.csv"
SUMMARY_CSV = SPLIT_DIR / "full_split_summary.csv"

for d in [SPLIT_DIR, AUG_IMAGE_DIR, AUG_LABELME_DIR, MASK_NPZ_CACHE_DIR, DEGRADED_IMAGE_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 验证集只从 origin 抽；新增 extra 永远 train_only。
VAL_RATIO_PER_CLASS = 0.20
MIN_VAL_PER_CLASS = 50
MAX_VAL_PER_CLASS = None  # 例如 250；None 表示不限制。
FORCE_RESCAN = True       # True: 每次重新扫描磁盘；但已存在 split 的旧样本 split 会保持不变。
FORCE_REBUILD_AUGMENT = False

# 模型专家。必须与 full_train.py 中 build_train_config() 的 domain_expert_names 完全一致。
EXPERT_NAMES = [
    "origin_competition",
    "open_fire_general",
    "indoor_kitchen_smoke",
    "manual_crop_normal_things",
    "traffic_vehicle_light",
    "natural_sky_weather",
    "artificial_light_product_media",
]


# 增强策略不是抽样比例。
# 所有真实图片都会进入训练；
# augment_policy 只决定是否额外生成增强图。
AUGMENT_POLICIES = {
    "none": [],

    # 原始比赛火灾样本
    "origin_fire": [
        "hflip",
        "jpeg_low",
        "blur_jpeg",
    ],

    # 原始比赛无火样本
    "origin_no_fire": [
        "hflip",
        "exposure_up",
        "blur_jpeg",
    ],

    # 普通清晰场景
    "general": [
        "hflip",
    ],

    # 灯光、天空、反射等困难负样本
    "hard_negative": [
        "hflip",
        "exposure_up",
        "blur_jpeg",
    ],

    # 低清监控、低质量车灯等
    "low_quality": [
        "hflip",
        "blur_jpeg",
    ],

    # 手工局部裁剪
    "manual_crop": [
        "hflip",
    ],

    # 新闻、视频截图、屏幕画面
    "news": [
        "hflip",
        "jpeg_low",
    ],

    # 商品展示图、素材图、孤立火焰图
    "product_media": [
        "hflip",
        "jpeg_low",
    ],
}


# 目录级数据配置：
# 1. 每个存在的目录全量进入训练，不做抽样。
# 2. augment_policy 只控制额外增强图。
# 3. expert 表示专家专业化阶段使用的 forced_expert。
# 4. 原始比赛数据通常由主数据加载器加载，
#    因此不在下面的 extra_datasets 配置中重复添加。
DATASET_SPECS = [
    # =========================================================
    # 通用明火与普通场景专家
    # =========================================================
    {
        "dir_name": "D-Fire fire",
        "label": 1,
        "expert": "open_fire_general",
        "augment_policy": "general",
        "sample_weight": 1.00,
    },
    {
        "dir_name": "D-Fire no_fire",
        "label": 0,
        "expert": "open_fire_general",
        "augment_policy": "general",
        "sample_weight": 1.00,
    },
    {
        "dir_name": "fireDetectVOCfinal Fire",
        "label": 1,
        "expert": "open_fire_general",
        "augment_policy": "general",
        "sample_weight": 1.00,
    },
    {
        "dir_name": "fireDetectVOCfinal NoFire",
        "label": 0,
        "expert": "open_fire_general",
        "augment_policy": "general",
        "sample_weight": 1.00,
    },
    {
        "dir_name": "同学收集的 fire",
        "label": 1,
        "expert": "open_fire_general",
        "augment_policy": "general",
        "sample_weight": 1.05,
    },
    {
        "dir_name": "同学收集的 no_fire",
        "label": 0,
        "expert": "open_fire_general",
        "augment_policy": "general",
        "sample_weight": 1.05,
    },
    {
        "dir_name": "困难训练集-gpt有火 fire",
        "label": 1,
        "expert": "open_fire_general",
        "augment_policy": "general",
        "sample_weight": 1.20,
    },
    {
        "dir_name": "困难训练集-gpt no_fire",
        "label": 0,
        "expert": "open_fire_general",
        "augment_policy": "hard_negative",
        "sample_weight": 1.20,
    },
    {
        "dir_name": "补充训练集-高清视频 video_fire",
        "label": 1,
        "expert": "open_fire_general",
        "augment_policy": "general",
        "sample_weight": 1.05,
    },
    {
        "dir_name": "补充训练集-高清视频 video_no_fire",
        "label": 0,
        "expert": "open_fire_general",
        "augment_policy": "general",
        "sample_weight": 1.05,
    },

    # =========================================================
    # 室内、厨房、烟雾和蒸汽专家
    # =========================================================
    {
        "dir_name": "Indoor Fire Smoke",
        "label": 1,
        "expert": "indoor_kitchen_smoke",
        "augment_policy": "general",
        "sample_weight": 1.15,
    },
    {
        "dir_name": "Indoor No Fire Smoke",
        "label": 0,
        "expert": "indoor_kitchen_smoke",
        "augment_policy": "hard_negative",
        "sample_weight": 1.20,
    },
    {
        "dir_name": "厨房监控 Fire",
        "label": 1,
        "expert": "indoor_kitchen_smoke",
        "augment_policy": "low_quality",
        "sample_weight": 1.30,
    },
    {
        "dir_name": "厨房监控 No Fire",
        "label": 0,
        "expert": "indoor_kitchen_smoke",
        "augment_policy": "low_quality",
        "sample_weight": 1.30,
    },

    # =========================================================
    # 局部裁剪、小目标、局部细节和普通图片专家
    # =========================================================
    {
        "dir_name": "manual_fire_crops",
        "label": 1,
        "expert": "manual_crop_normal_things",
        "augment_policy": "manual_crop",
        "sample_weight": 1.25,
    },
    {
        "dir_name": "manual_no_fire_crops",
        "label": 0,
        "expert": "manual_crop_normal_things",
        "augment_policy": "manual_crop",
        "sample_weight": 1.25,
    },
    {
        "dir_name": "小火焰 fire",
        "label": 1,
        "expert": "manual_crop_normal_things",
        "augment_policy": "hflip",
        "sample_weight": 1.25,
    },
        {
        "dir_name": "毫不相干的 no_fire",
        "label": 1,
        "expert": "manual_crop_normal_things",
        "augment_policy": "hflip",
        "sample_weight": 1.25,
    },

    # =========================================================
    # 车辆、车灯、道路与交通设施专家
    # =========================================================
    {
        "dir_name": "Night car light",
        "label": 0,
        "expert": "traffic_vehicle_light",
        "augment_policy": "hard_negative",
        "sample_weight": 1.25,
    },
    {
        "dir_name": "交通无火训练集",
        "label": 0,
        "expert": "traffic_vehicle_light",
        "augment_policy": "hard_negative",
        "sample_weight": 1.25,
    },
    {
        "dir_name": "低质量车灯",
        "label": 0,
        "expert": "traffic_vehicle_light",
        "augment_policy": "low_quality",
        "sample_weight": 1.35,
    },

    # =========================================================
    # 日出、夕阳、火烧云、闪电和自然天空专家
    # =========================================================
    {
        "dir_name": "Time Of Day Dataset Sunrise",
        "label": 0,
        "expert": "natural_sky_weather",
        "augment_policy": "hard_negative",
        "sample_weight": 1.25,
    },
    {
        "dir_name": "火烧云",
        "label": 0,
        "expert": "natural_sky_weather",
        "augment_policy": "hard_negative",
        "sample_weight": 1.30,
    },
    {
        "dir_name": "困难训练集-闪电 no_fire",
        "label": 0,
        "expert": "natural_sky_weather",
        "augment_policy": "hard_negative",
        "sample_weight": 1.30,
    },

    # =========================================================
    # 人工光源、LED、反光、商品图与媒体画面专家
    # =========================================================
    {
        "dir_name": "LED 反光",
        "label": 0,
        "expert": "artificial_light_product_media",
        "augment_policy": "hard_negative",
        "sample_weight": 1.30,
    },
    {
        "dir_name": "困难训练集-诡异光照 no_fire",
        "label": 0,
        "expert": "artificial_light_product_media",
        "augment_policy": "hard_negative",
        "sample_weight": 1.35,
    },

    # 网图 fire 是商品图片，不属于普通真实场景明火。
    {
        "dir_name": "网图 fire",
        "label": 1,
        "expert": "artificial_light_product_media",
        "augment_policy": "product_media",
        "sample_weight": 1.15,
    },

    # 新闻和媒体画面
    {
        "dir_name": "新闻报道 fire",
        "label": 1,
        "expert": "artificial_light_product_media",
        "augment_policy": "news",
        "sample_weight": 1.15,
    },
    {
        "dir_name": "新闻报道 no fire",
        "label": 0,
        "expert": "artificial_light_product_media",
        "augment_policy": "news",
        "sample_weight": 1.20,
    },

    # 火焰视频、素材、合成画面
    {
        "dir_name": "火焰视频素材 fire",
        "label": 1,
        "expert": "artificial_light_product_media",
        "augment_policy": "product_media",
        "sample_weight": 1.15,
    },
]

# LabelMe mask 标签。可按你自己的标注名继续补充。
POSITIVE_LABELS = {
    "fire", "flame", "flames", "smoke_fire", "fire_smoke", "burning", "火", "火焰", "着火", "明火",
}
NEGATIVE_LABELS = {
    "light", "lamp", "led", "reflection", "sunset", "sunrise", "cloud", "headlight", "car_light",
    "text", "subtitle", "watermark", "smoke", "no_fire", "negative", "反光", "车灯", "灯光", "火烧云", "日落", "日出",
}

# 降质缓存策略。正类不要过猛；负类和 low_quality/hard_negative 可以更强。
DEGRADED_FIRE_STAGES = [2, 3, 4, 5, 6]
DEGRADED_NO_FIRE_STAGES = [3, 4, 5, 8, 9, 10]
DEGRADED_LOW_QUALITY_NO_FIRE_STAGES = [4, 5, 8, 9, 10, 11, 12]
MAX_DEGRADED_VARIANTS_PER_IMAGE = 3
CACHE_NUM_WORKERS = max(1, min(8, (os.cpu_count() or 2) - 1))
AUG_NUM_WORKERS = CACHE_NUM_WORKERS  # 增强图片生成线程数；I/O 较多，通常和缓存线程数一致即可。

print("专家数量:", len(EXPERT_NAMES), EXPERT_NAMES)
print("extra_datasets:", EXTRA_DATASET_DIR)

In [ ]:
# =========================
# Cell 2 - 全量划分 / 增量检测
# =========================


def clean_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    text = str(value).strip()
    if text.lower() in {"", "nan", "none", "null"}:
        return ""
    return text


def rel_path(path: Path) -> str:
    path = Path(path).resolve()
    try:
        return str(path.relative_to(ROOT)).replace("\\", "/")
    except Exception:
        return str(path).replace("\\", "/")


def stable_hash_text(text: str) -> str:
    return hashlib.sha1(str(text).replace("\\", "/").encode("utf-8")).hexdigest()


def stable_float(text: str) -> float:
    return int(stable_hash_text(text)[:12], 16) / float(16 ** 12)


def image_fingerprint(path: Path, label: int, dataset_key: str) -> str:
    path = Path(path)
    stat = path.stat()
    base = f"{rel_path(path)}|label={int(label)}|dataset={dataset_key}|size={stat.st_size}"
    return stable_hash_text(base)[:24]


def list_images(root_dirs):
    rows = []
    seen = set()
    for d in root_dirs:
        d = Path(d)
        if not d.exists():
            continue
        for p in tqdm(sorted(d.rglob("*")), desc=f"扫描 {d.name}", unit="file", leave=False):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                key = str(p.resolve())
                if key not in seen:
                    seen.add(key)
                    rows.append(p)
    return rows


def find_adjacent_json(image_path: Path):
    direct = image_path.with_suffix(".json")
    if direct.exists():
        return direct
    parent = image_path.parent
    matches = list(parent.rglob(f"{image_path.stem}.json"))
    return matches[0] if matches else None


def safe_image_size(path: Path):
    try:
        with Image.open(path) as im:
            return im.size
    except Exception:
        return (np.nan, np.nan)


def make_row(path: Path, label: int, dataset_key: str, source_dataset: str, expert_name: str, augment_policy: str, sample_weight: float, is_origin: bool):
    w, h = safe_image_size(path)
    ann = find_adjacent_json(path)
    sid = image_fingerprint(path, label, dataset_key)
    return {
        "sample_id": sid,
        "path": "",  # 训练优先使用 relative_path；保留空绝对路径，方便迁移。
        "relative_path": rel_path(path),
        "label": int(label),
        "expert_name": str(expert_name),
        "dataset_key": str(dataset_key),
        "source_dataset": str(source_dataset),
        "source_type": "origin" if is_origin else f"extra::{source_dataset}",
        "is_origin": bool(is_origin),
        "is_augmented": False,
        "aug_type": "",
        "augment_policy": str(augment_policy),
        "sample_weight": float(sample_weight),
        "annotation_path": "",
        "relative_annotation_path": rel_path(ann) if ann else "",
        "width": w,
        "height": h,
    }


def scan_current_registry() -> pd.DataFrame:
    rows = []
    fire_dirs = [d for d in FIRE_IMAGE_DIRS if Path(d).exists()]
    no_fire_dirs = [d for d in NO_FIRE_IMAGE_DIRS if Path(d).exists()]

    for p in list_images(fire_dirs):
        rows.append(make_row(p, 1, "origin", "origin_fire", "origin_competition", "origin_fire", 1.00, True))
    for p in list_images(no_fire_dirs):
        rows.append(make_row(p, 0, "origin", "origin_no_fire", "origin_competition", "origin_no_fire", 1.00, True))

    for spec in tqdm(DATASET_SPECS, desc="扫描 extra_datasets", unit="dataset"):
        d = EXTRA_DATASET_DIR / spec["dir_name"]
        if not d.exists():
            print(f"[跳过] 目录不存在: {d}")
            continue
        if spec["expert"] not in EXPERT_NAMES:
            raise ValueError(f"未知 expert: {spec['expert']} in {spec['dir_name']}")
        for p in list_images([d]):
            rows.append(make_row(
                p,
                int(spec["label"]),
                f"extra::{spec['dir_name']}",
                spec["dir_name"],
                spec["expert"],
                spec.get("augment_policy", "none"),
                float(spec.get("sample_weight", 1.0)),
                False,
            ))

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise RuntimeError("没有扫描到任何图片，请检查 ROOT、origin 路径和 extra_datasets 配置。")
    df = df.drop_duplicates(subset=["sample_id"]).reset_index(drop=True)
    return df


def build_or_update_split_manifest(current_df: pd.DataFrame, previous_registry: pd.DataFrame) -> pd.DataFrame:
    """优先按旧 sample_id 保持划分；sample_id 因文件大小变化时按路径保持。"""
    if SPLIT_MANIFEST_CSV.exists():
        old = pd.read_csv(SPLIT_MANIFEST_CSV)
        old_map = dict(zip(old["sample_id"].astype(str), old["split"].astype(str)))
    else:
        old = pd.DataFrame(columns=["sample_id", "split"])
        old_map = {}

    old_path_map = {}
    if "relative_path" in old.columns:
        old_path_map.update({
            clean_text(path).replace("\\", "/"): str(split)
            for path, split in zip(old["relative_path"], old["split"])
            if clean_text(path)
        })
    elif len(previous_registry) > 0 and {"sample_id", "relative_path"}.issubset(previous_registry.columns):
        registry_view = previous_registry[["sample_id", "relative_path"]].copy()
        registry_view["sample_id"] = registry_view["sample_id"].astype(str)
        old_with_path = old.merge(registry_view, on="sample_id", how="left")
        old_path_map.update({
            clean_text(path).replace("\\", "/"): str(split)
            for path, split in zip(old_with_path["relative_path"], old_with_path["split"])
            if clean_text(path)
        })

    current_ids = set(current_df["sample_id"].astype(str))
    old_ids = set(old_map)
    new_ids = sorted(current_ids - old_ids)
    missing_ids = sorted(old_ids - current_ids)
    print(f"当前样本: {len(current_ids)} | 历史样本: {len(old_ids)} | 新增ID: {len(new_ids)} | 本次未扫描到历史ID: {len(missing_ids)}")

    rows = []
    for record in tqdm(current_df.to_dict("records"), total=len(current_df), desc="生成/保持 split", unit="img"):
        sid = str(record["sample_id"])
        path_key = clean_text(record.get("relative_path", "")).replace("\\", "/")
        if sid in old_map:
            split = old_map[sid]
        elif path_key in old_path_map:
            # 原图路径不变时，即使文件大小变化导致 sample_id 变化，也保持原划分。
            split = old_path_map[path_key]
        elif bool(record["is_origin"]):
            split = "val_candidate"
        else:
            split = "train"
        rows.append({"sample_id": sid, "relative_path": path_key, "split": split})

    manifest = pd.DataFrame(rows)

    # 对真正新增的 origin 候选做按 label 的稳定验证集分配；旧路径的 split 保持不动。
    merged = current_df.merge(manifest[["sample_id", "split"]], on="sample_id", how="left")
    for label in [0, 1]:
        mask = (merged["split"] == "val_candidate") & (merged["is_origin"] == True) & (merged["label"].astype(int) == label)
        idx = list(merged.index[mask])
        if not idx:
            continue
        idx_sorted = sorted(idx, key=lambda i: stable_float(merged.at[i, "sample_id"]))
        target_n = max(int(round(len(idx_sorted) * float(VAL_RATIO_PER_CLASS))), int(MIN_VAL_PER_CLASS) if len(idx_sorted) >= int(MIN_VAL_PER_CLASS) else 1)
        if MAX_VAL_PER_CLASS is not None:
            target_n = min(target_n, int(MAX_VAL_PER_CLASS))
        target_n = min(max(target_n, 1), len(idx_sorted))
        val_set = set(idx_sorted[:target_n])
        split_updates = {str(merged.at[i, "sample_id"]): ("val" if i in val_set else "train") for i in idx_sorted}
        manifest.loc[manifest["sample_id"].isin(split_updates), "split"] = manifest.loc[
            manifest["sample_id"].isin(split_updates), "sample_id"
        ].map(split_updates)

    manifest["split"] = manifest["split"].replace({"val_candidate": "train"})
    return manifest[["sample_id", "relative_path", "split"]].copy()


previous_registry_df = pd.read_csv(DATASET_REGISTRY_CSV) if DATASET_REGISTRY_CSV.exists() else pd.DataFrame()
current_df = scan_current_registry()
manifest_df = build_or_update_split_manifest(current_df, previous_registry_df)
# 先生成路径回退映射，再覆盖当前 registry。
current_df.to_csv(DATASET_REGISTRY_CSV, index=False, encoding="utf-8-sig")
manifest_df.to_csv(SPLIT_MANIFEST_CSV, index=False, encoding="utf-8-sig")

split_df = current_df.merge(manifest_df[["sample_id", "split"]], on="sample_id", how="left")
split_df["split"] = split_df["split"].fillna("train")

# extra 数据强制 train，避免污染 origin-only 验证集。
split_df.loc[split_df["is_origin"] == False, "split"] = "train"

train_real_df = split_df[split_df["split"] == "train"].copy().reset_index(drop=True)
val_df = split_df[(split_df["split"] == "val") & (split_df["is_origin"] == True)].copy().reset_index(drop=True)
test_df = split_df.iloc[0:0].copy()
test_df["split"] = []

train_real_df.to_csv(TRAIN_REAL_CSV, index=False, encoding="utf-8-sig")
# Cell 3 会加入增强图并重写 TRAIN_ALL_CSV；这里先写一份 real-only 版本，方便调试。
train_real_df.to_csv(TRAIN_ALL_CSV, index=False, encoding="utf-8-sig")
val_df.to_csv(VAL_CSV, index=False, encoding="utf-8-sig")
test_df.to_csv(TEST_CSV, index=False, encoding="utf-8-sig")

summary = []
for name, df in [("train_real", train_real_df), ("val_origin", val_df), ("test_empty", test_df)]:
    for (expert, label), g in df.groupby(["expert_name", "label"], dropna=False):
        summary.append({"split": name, "expert_name": expert, "label": int(label), "count": len(g)})
summary_df = pd.DataFrame(summary)
summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

print("\n训练 real-only 分布:")
print(train_real_df.groupby(["expert_name", "label"], dropna=False).size())
print("\n验证集分布，只允许 origin:")
print(val_df.groupby(["source_dataset", "label"], dropna=False).size())
print("\n输出:")
print("registry:", DATASET_REGISTRY_CSV)
print("split_manifest:", SPLIT_MANIFEST_CSV)
print("train_real:", TRAIN_REAL_CSV)
print("train_all 初始 real-only:", TRAIN_ALL_CSV)
print("val:", VAL_CSV)

In [ ]:
# =========================
# Cell 3 - 生成增强图片、mask/image npz 缓存、离线降质缓存
# =========================


def resolve_rel(relative_path: str) -> Path:
    p = ROOT / clean_text(relative_path)
    if not p.exists():
        raise FileNotFoundError(f"找不到文件: {p}")
    return p


def load_labelme(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_labelme(data, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def transform_labelme_for_aug(source_json: Path, aug_type: str, aug_image_path: Path):
    data = deepcopy(load_labelme(source_json))
    w = int(data.get("imageWidth", 0) or 0)
    h = int(data.get("imageHeight", 0) or 0)
    if w <= 0 or h <= 0:
        try:
            with Image.open(ROOT / rel_path(aug_image_path)) as im:
                w, h = im.size
        except Exception:
            pass
    if aug_type == "hflip":
        for shape in data.get("shapes", []):
            pts = []
            for x, y in shape.get("points", []):
                pts.append([float(max(0, min(w, w - float(x)))), float(max(0, min(h, float(y))))])
            shape["points"] = pts
    # 非几何增强直接复制标注。
    data["imagePath"] = aug_image_path.name
    data["imageData"] = None
    data["imageWidth"] = int(w)
    data["imageHeight"] = int(h)
    data.setdefault("flags", {})
    data["flags"]["aug_type"] = str(aug_type)
    data["flags"]["source_json"] = rel_path(source_json)
    return data


def apply_augment(image: Image.Image, aug_type: str) -> Image.Image:
    if aug_type == "hflip":
        return ImageOps.mirror(image)
    if aug_type == "exposure_up":
        return ImageEnhance.Brightness(image).enhance(1.18)
    if aug_type == "jpeg_low":
        buf = io.BytesIO()
        image.save(buf, format="JPEG", quality=55, optimize=False, progressive=False, subsampling=2)
        buf.seek(0)
        return Image.open(buf).convert("RGB")
    if aug_type == "blur_jpeg":
        out = image.filter(ImageFilter.GaussianBlur(0.65))
        buf = io.BytesIO()
        out.save(buf, format="JPEG", quality=48, optimize=False, progressive=False, subsampling=2)
        buf.seek(0)
        return Image.open(buf).convert("RGB")
    raise ValueError(f"未知 aug_type: {aug_type}")


def make_augmented_rows(train_real_df: pd.DataFrame) -> pd.DataFrame:
    """
    多线程生成增强图片。

    这里生成的是“离线增强样本”：会把增强后的图片真正写到
    fire_splits/augmented_images_full，并把增强样本追加到 train_all.csv。
    """

    def make_augmented_rows_for_one(record: dict):
        local_rows = []
        local_errors = []
        policy = clean_text(record.get("augment_policy", "none")) or "none"
        aug_types = list(AUGMENT_POLICIES.get(policy, []))
        if not aug_types:
            return local_rows, local_errors

        rel_src = clean_text(record.get("relative_path", ""))
        try:
            src = resolve_rel(rel_src)
        except Exception as exc:
            local_errors.append(f"[跳过增强] 路径失败: {rel_src} ({exc})")
            return local_rows, local_errors

        try:
            with Image.open(src) as im:
                image = im.convert("RGB")
        except Exception as exc:
            local_errors.append(f"[跳过增强] 读取失败: {src} ({exc})")
            return local_rows, local_errors

        label_value = int(record.get("label", 0))
        subdir = "fire" if label_value == 1 else "no_fire"
        source_json_rel = clean_text(record.get("relative_annotation_path", ""))
        source_json = ROOT / source_json_rel if source_json_rel else None

        for aug_type in aug_types:
            try:
                key = stable_hash_text(f"{rel_src}|{aug_type}|{label_value}")[:24]
                aug_path = AUG_IMAGE_DIR / subdir / f"{key}_{aug_type}.jpg"
                if FORCE_REBUILD_AUGMENT or not aug_path.exists():
                    aug_path.parent.mkdir(parents=True, exist_ok=True)
                    out = apply_augment(image, aug_type)
                    out.save(aug_path, format="JPEG", quality=92)

                aug_json_rel = ""
                if source_json is not None and source_json.exists():
                    aug_json = AUG_LABELME_DIR / subdir / f"{key}_{aug_type}.json"
                    if FORCE_REBUILD_AUGMENT or not aug_json.exists():
                        try:
                            aug_json.parent.mkdir(parents=True, exist_ok=True)
                            aug_data = transform_labelme_for_aug(source_json, aug_type, aug_path)
                            save_labelme(aug_data, aug_json)
                        except Exception as exc:
                            local_errors.append(f"[警告] 增强标注转换失败: {source_json} ({exc})")
                            aug_json = None
                    if aug_json is not None:
                        aug_json_rel = rel_path(aug_json)

                aug_row = dict(record)
                aug_row["sample_id"] = f"aug_{key}"
                aug_row["path"] = ""
                aug_row["relative_path"] = rel_path(aug_path)
                aug_row["is_augmented"] = True
                aug_row["aug_type"] = aug_type
                aug_row["source_type"] = f"aug::{record.get('source_type', '')}"
                aug_row["relative_annotation_path"] = aug_json_rel
                aug_row["annotation_path"] = ""
                aug_row["sample_weight"] = float(record.get("sample_weight", 1.0))
                local_rows.append(aug_row)
            except Exception as exc:
                local_errors.append(f"[跳过增强] {rel_src} aug_type={aug_type} ({exc})")

        return local_rows, local_errors

    records = train_real_df.to_dict("records")
    aug_rows = []
    aug_errors = []
    (AUG_IMAGE_DIR / "fire").mkdir(parents=True, exist_ok=True)
    (AUG_IMAGE_DIR / "no_fire").mkdir(parents=True, exist_ok=True)
    (AUG_LABELME_DIR / "fire").mkdir(parents=True, exist_ok=True)
    (AUG_LABELME_DIR / "no_fire").mkdir(parents=True, exist_ok=True)

    with ThreadPoolExecutor(max_workers=AUG_NUM_WORKERS) as ex:
        futures = [ex.submit(make_augmented_rows_for_one, record) for record in records]
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"生成增强图片({AUG_NUM_WORKERS}线程)", unit="img"):
            try:
                local_rows, local_errors = fut.result()
                aug_rows.extend(local_rows)
                aug_errors.extend(local_errors)
            except Exception as exc:
                aug_errors.append(f"[增强线程失败] {exc}")

    if aug_errors:
        print(f"增强阶段共有 {len(aug_errors)} 条警告/错误，前 20 条如下：")
        for msg in aug_errors[:20]:
            print(msg)
        if len(aug_errors) > 20:
            print(f"... 其余 {len(aug_errors) - 20} 条已省略")

    return pd.DataFrame(aug_rows)


def normalize_relative_path(value: str) -> str:
    text = clean_text(value).replace("\\", "/")
    while text.startswith("./"):
        text = text[2:]
    return text.strip("/")


def load_new_cache_index(path: Path, required_columns, cache_name: str) -> pd.DataFrame:
    required_columns = list(required_columns)
    if not path.exists():
        return pd.DataFrame(columns=required_columns)
    df = pd.read_csv(path)
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise RuntimeError(
            f"{cache_name} 是旧缓存格式，缺少字段 {missing}: {path}。"
            "请先运行 migrate_cache_to_path_index.py，再执行本 Notebook。"
        )
    return df


def mask_cache_key_for_path(relative_path: str) -> str:
    return stable_hash_text(normalize_relative_path(relative_path))[:24]


def mask_npz_path_for(relative_path: str) -> Path:
    return MASK_NPZ_CACHE_DIR / f"{mask_cache_key_for_path(relative_path)}.npz"


def save_npz_atomic(path: Path, **arrays) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(path.name + f".{os.getpid()}.tmp.npz")
    try:
        np.savez_compressed(temp_path, **arrays)
        os.replace(temp_path, path)
    finally:
        if temp_path.exists():
            temp_path.unlink(missing_ok=True)


def draw_labelme_shape(mask: Image.Image, shape: dict, src_w: int, src_h: int, dst_w: int, dst_h: int):
    points = shape.get("points", [])
    if not points:
        return
    sx = dst_w / max(float(src_w), 1.0)
    sy = dst_h / max(float(src_h), 1.0)
    pts = [(max(0, min(dst_w - 1, float(x) * sx)), max(0, min(dst_h - 1, float(y) * sy))) for x, y in points]
    draw = ImageDraw.Draw(mask)
    shape_type = str(shape.get("shape_type", "polygon")).lower()
    if shape_type == "rectangle" and len(pts) >= 2:
        x1, y1 = pts[0]
        x2, y2 = pts[1]
        draw.rectangle([min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)], fill=1)
    elif len(pts) >= 3:
        draw.polygon(pts, fill=1)
    elif len(pts) == 2:
        x1, y1 = pts[0]
        x2, y2 = pts[1]
        draw.rectangle([min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)], fill=1)


def make_mask_cache_record(record: dict):
    relative_path = normalize_relative_path(record["relative_path"])
    src = resolve_rel(relative_path)
    mask_npz = mask_npz_path_for(relative_path)
    warning = ""
    with Image.open(src) as source_image:
        w, h = source_image.size

    pos = Image.new("L", (w, h), 0)
    neg = Image.new("L", (w, h), 0)
    pos_count = 0
    neg_count = 0
    ann_rel = normalize_relative_path(record.get("relative_annotation_path", ""))
    if ann_rel:
        ann = ROOT / ann_rel
        if ann.exists():
            try:
                data = load_labelme(ann)
                src_w = int(data.get("imageWidth", w) or w)
                src_h = int(data.get("imageHeight", h) or h)
                for shape in data.get("shapes", []):
                    label = str(shape.get("label", "")).lower().strip()
                    if label in POSITIVE_LABELS:
                        draw_labelme_shape(pos, shape, src_w, src_h, w, h)
                        pos_count += 1
                    elif label in NEGATIVE_LABELS:
                        draw_labelme_shape(neg, shape, src_w, src_h, w, h)
                        neg_count += 1
            except Exception as exc:
                warning = f"annotation_read_failed:{exc}"
    if int(record.get("label", 0)) == 0 and pos_count > 0:
        warning = "label_0_has_positive_annotation"

    # 只有索引中没有该路径时才会调用本函数；不检查旧文件内容或时间。
    save_npz_atomic(
        mask_npz,
        positive_mask=np.asarray(pos, dtype=np.uint8),
        negative_mask=np.asarray(neg, dtype=np.uint8),
        positive_region_count=int(pos_count),
        negative_region_count=int(neg_count),
        original_width=int(w),
        original_height=int(h),
        annotation_warning=warning,
    )
    return {
        "cache_key": mask_cache_key_for_path(relative_path),
        "relative_original_path": relative_path,
        "relative_mask_npz_path": rel_path(mask_npz),
        "has_positive_mask": bool(pos_count > 0 and int(record.get("label", 0)) == 1),
        "has_negative_mask": bool(neg_count > 0),
        "annotation_warning": warning,
        "original_width": int(w),
        "original_height": int(h),
    }


def attach_mask_index(df: pd.DataFrame, mask_index_df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    old_cache_columns = [
        "relative_image_npz_path", "image_npz_path",
        "relative_mask_npz_path", "mask_npz_path",
        "has_positive_mask", "has_negative_mask", "annotation_warning",
        "mask_cache_key",
    ]
    out.drop(columns=[c for c in old_cache_columns if c in out.columns], inplace=True)
    out["_cache_path_key"] = out["relative_path"].map(normalize_relative_path)

    merge_columns = [
        "cache_key", "relative_original_path", "relative_mask_npz_path",
        "has_positive_mask", "has_negative_mask", "annotation_warning",
    ]
    cache_view = mask_index_df[merge_columns].copy()
    cache_view["_cache_path_key"] = cache_view["relative_original_path"].map(normalize_relative_path)
    cache_view = cache_view.drop_duplicates(subset=["_cache_path_key"], keep="first")
    cache_view.rename(columns={"cache_key": "mask_cache_key"}, inplace=True)
    cache_view.drop(columns=["relative_original_path"], inplace=True)

    out = out.merge(cache_view, on="_cache_path_key", how="left", sort=False, validate="m:1")
    out.drop(columns=["_cache_path_key"], inplace=True)
    out["relative_mask_npz_path"] = out["relative_mask_npz_path"].fillna("")
    out["mask_cache_key"] = out["mask_cache_key"].fillna("")
    out["has_positive_mask"] = out["has_positive_mask"].fillna(False).astype(bool)
    out["has_negative_mask"] = out["has_negative_mask"].fillna(False).astype(bool)
    out["annotation_warning"] = out["annotation_warning"].fillna("")
    return out


def degradation_params_for_stage(stage: int, label: int):
    table = {
        0: {"jpeg_quality": 92},
        1: {"jpeg_quality": 82},
        2: {"jpeg_quality": 72},
        3: {"downscale": 0.84, "jpeg_quality": 68},
        4: {"downscale": 0.72, "blur": 0.40, "jpeg_quality": 61},
        5: {"downscale": 0.61, "blur": 0.65, "jpeg_quality": 54, "saturation": 1.03},
        6: {"downscale": 0.52, "blur": 0.95, "jpeg_quality": 48},
        8: {"downscale": 0.40, "blur": 1.35, "jpeg_quality": 38, "saturation": 1.06},
        9: {"downscale": 0.33, "blur": 1.80, "jpeg_quality": 31, "contrast": 0.93},
        10: {"downscale": 0.27, "blur": 2.30, "jpeg_quality": 25, "saturation": 1.09},
        11: {"downscale": 0.22, "blur": 3.00, "jpeg_quality": 18, "contrast": 0.87},
        12: {"downscale": 0.17, "blur": 3.70, "jpeg_quality": 13, "contrast": 0.83},
    }
    params = dict(table.get(int(stage), table[0]))
    if int(label) == 1:
        if "blur" in params:
            params["blur"] = min(float(params["blur"]), 0.85)
        if "downscale" in params:
            params["downscale"] = max(float(params["downscale"]), 0.50)
        params["jpeg_quality"] = max(float(params.get("jpeg_quality", 85)), 42.0)
    return params


def apply_degradation(image: Image.Image, stage: int, label: int):
    params = degradation_params_for_stage(stage, label)
    out = image
    if "downscale" in params:
        scale = float(params["downscale"])
        w, h = out.size
        small = out.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.Resampling.BICUBIC)
        out = small.resize((w, h), Image.Resampling.BILINEAR)
    if "blur" in params:
        out = out.filter(ImageFilter.GaussianBlur(float(params["blur"])))
    if "saturation" in params:
        out = ImageEnhance.Color(out).enhance(float(params["saturation"]))
    if "contrast" in params:
        out = ImageEnhance.Contrast(out).enhance(float(params["contrast"]))
    buf = io.BytesIO()
    out.save(buf, format="JPEG", quality=int(round(float(params.get("jpeg_quality", 85)))), optimize=False, progressive=False, subsampling=2)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def stages_for_row(row):
    label = int(row.get("label", 0))
    expert = clean_text(row.get("expert_name", ""))
    policy = clean_text(row.get("augment_policy", ""))
    if label == 1:
        stages = DEGRADED_FIRE_STAGES
    elif expert == "ultra_low_quality_blurry" or policy == "low_quality":
        stages = DEGRADED_LOW_QUALITY_NO_FIRE_STAGES
    else:
        stages = DEGRADED_NO_FIRE_STAGES
    # 仍用 sample_id 稳定选择 stage，保证原有每张图的阶段划分不变。
    rng = random.Random(stable_hash_text(str(row["sample_id"])))
    stages = list(stages)
    rng.shuffle(stages)
    return sorted(stages[:MAX_DEGRADED_VARIANTS_PER_IMAGE])


def degraded_cache_key(relative_path: str, stage: int) -> str:
    return f"{stable_hash_text(normalize_relative_path(relative_path))[:24]}:stage={int(stage):02d}"


def degraded_path_for(relative_path: str, label: int, stage: int) -> Path:
    path_hash = stable_hash_text(normalize_relative_path(relative_path))[:24]
    subdir = "fire" if int(label) == 1 else "no_fire"
    return DEGRADED_IMAGE_CACHE_DIR / subdir / f"{path_hash}_stage{int(stage):02d}.jpg"


def make_degraded_for_record(record: dict):
    relative_path = normalize_relative_path(record["relative_path"])
    label = int(record["label"])
    pending_stages = [int(stage) for stage in record.get("_pending_stages", [])]
    if not pending_stages:
        return []
    src = resolve_rel(relative_path)
    try:
        with Image.open(src) as source_image:
            image = source_image.convert("RGB")
    except Exception as exc:
        return [{"error": f"read_failed:{exc}", "relative_original_path": relative_path}]

    out_rows = []
    for stage in pending_stages:
        out_path = degraded_path_for(relative_path, label, stage)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        degraded = apply_degradation(image, stage, label)
        degraded.save(out_path, format="JPEG", quality=92)
        out_rows.append({
            "cache_key": degraded_cache_key(relative_path, stage),
            "relative_original_path": relative_path,
            "relative_degraded_path": rel_path(out_path),
            "label": label,
            "degradation_stage": int(stage),
            "sample_id": record.get("sample_id", ""),
            "expert_name": record.get("expert_name", ""),
            "source_dataset": record.get("source_dataset", ""),
        })
    return out_rows


train_real_df = pd.read_csv(TRAIN_REAL_CSV)
aug_df = make_augmented_rows(train_real_df)
if len(aug_df) > 0:
    train_all_df = pd.concat([train_real_df, aug_df], ignore_index=True)
else:
    train_all_df = train_real_df.copy()

# 先写入未带缓存字段的 train_all，随后按路径索引补充 mask 字段。
train_all_df.to_csv(TRAIN_ALL_CSV, index=False, encoding="utf-8-sig")
print(f"真实训练图: {len(train_real_df)} | 增强图: {len(aug_df)} | train_all: {len(train_all_df)}")

# -------------------------
# 路径键 mask 缓存
# -------------------------
mask_columns = [
    "cache_key", "relative_original_path", "relative_mask_npz_path",
    "has_positive_mask", "has_negative_mask", "annotation_warning",
    "original_width", "original_height",
]
mask_index_df = load_new_cache_index(MASK_CACHE_INDEX_CSV, mask_columns, "mask 缓存索引")
if len(mask_index_df) > 0:
    mask_index_df["relative_original_path"] = mask_index_df["relative_original_path"].map(normalize_relative_path)
    mask_index_df = mask_index_df.drop_duplicates(subset=["relative_original_path"], keep="first").reset_index(drop=True)
completed_mask_paths = set(mask_index_df["relative_original_path"].astype(str))

cache_input = pd.concat([train_all_df, pd.read_csv(VAL_CSV)], ignore_index=True)
cache_input["relative_path"] = cache_input["relative_path"].map(normalize_relative_path)
cache_input = cache_input.drop_duplicates(subset=["relative_path"], keep="first").reset_index(drop=True)
pending_mask_df = cache_input[~cache_input["relative_path"].isin(completed_mask_paths)].copy()

new_mask_rows = []
mask_records = pending_mask_df.to_dict("records")
with ThreadPoolExecutor(max_workers=CACHE_NUM_WORKERS) as ex:
    futures = [ex.submit(make_mask_cache_record, record) for record in mask_records]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="生成路径键 mask 缓存", unit="img"):
        try:
            new_mask_rows.append(fut.result())
        except Exception as exc:
            print("[mask 缓存失败]", exc)

if new_mask_rows:
    mask_index_df = pd.concat([mask_index_df, pd.DataFrame(new_mask_rows)], ignore_index=True)
mask_index_df = mask_index_df.drop_duplicates(subset=["relative_original_path"], keep="first").reset_index(drop=True)
mask_index_df.to_csv(MASK_CACHE_INDEX_CSV, index=False, encoding="utf-8-sig")

train_all_df = attach_mask_index(train_all_df, mask_index_df)
val_df = attach_mask_index(pd.read_csv(VAL_CSV), mask_index_df)
train_all_df.to_csv(TRAIN_ALL_CSV, index=False, encoding="utf-8-sig")
val_df.to_csv(VAL_CSV, index=False, encoding="utf-8-sig")
print(f"mask 索引记录: {len(mask_index_df)} | 本次新增: {len(new_mask_rows)}")

# -------------------------
# 路径 + stage 降质缓存
# -------------------------
degraded_columns = [
    "cache_key", "relative_original_path", "relative_degraded_path",
    "label", "degradation_stage", "sample_id", "expert_name", "source_dataset",
]
deg_index_df = load_new_cache_index(DEGRADED_INDEX_CSV, degraded_columns, "降质缓存索引")
if len(deg_index_df) > 0:
    deg_index_df["relative_original_path"] = deg_index_df["relative_original_path"].map(normalize_relative_path)
    deg_index_df["degradation_stage"] = pd.to_numeric(deg_index_df["degradation_stage"], errors="coerce").fillna(-1).astype(int)
    deg_index_df = deg_index_df.drop_duplicates(
        subset=["relative_original_path", "degradation_stage"], keep="first"
    ).reset_index(drop=True)
completed_degraded = set(zip(
    deg_index_df["relative_original_path"].astype(str),
    deg_index_df["degradation_stage"].astype(int),
))

deg_tasks = []
for record in train_all_df.to_dict("records"):
    relative_path = normalize_relative_path(record.get("relative_path", ""))
    pending_stages = [
        int(stage) for stage in stages_for_row(record)
        if (relative_path, int(stage)) not in completed_degraded
    ]
    if pending_stages:
        record["relative_path"] = relative_path
        record["_pending_stages"] = pending_stages
        deg_tasks.append(record)

new_deg_rows = []
with ThreadPoolExecutor(max_workers=CACHE_NUM_WORKERS) as ex:
    futures = [ex.submit(make_degraded_for_record, record) for record in deg_tasks]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="生成路径键 degraded 缓存", unit="img"):
        try:
            result = fut.result()
            if isinstance(result, list):
                new_deg_rows.extend([row for row in result if "relative_degraded_path" in row])
                for row in result:
                    if "error" in row:
                        print("[降质缓存失败]", row)
        except Exception as exc:
            print("[降质缓存失败]", exc)

if new_deg_rows:
    deg_index_df = pd.concat([deg_index_df, pd.DataFrame(new_deg_rows)], ignore_index=True)
deg_index_df = deg_index_df.drop_duplicates(
    subset=["relative_original_path", "degradation_stage"], keep="first"
).reset_index(drop=True)
deg_index_df.to_csv(DEGRADED_INDEX_CSV, index=False, encoding="utf-8-sig")
print(f"降质索引记录: {len(deg_index_df)} | 本次新增 variant: {len(new_deg_rows)}")

# 统计与泄漏检查。
print("\n最终 train_all 分布:")
print(train_all_df.groupby(["expert_name", "label", "is_augmented"], dropna=False).size())
print("\n最终 val 分布:")
print(val_df.groupby(["source_dataset", "label"], dropna=False).size())
print("\nmask 统计:")
for name, df in [("train_all", train_all_df), ("val", val_df)]:
    pos = int((df.get("has_positive_mask", pd.Series([], dtype=bool)) == True).sum()) if "has_positive_mask" in df.columns else 0
    neg = int((df.get("has_negative_mask", pd.Series([], dtype=bool)) == True).sum()) if "has_negative_mask" in df.columns else 0
    print(name, "positive_mask_rows=", pos, "negative_mask_rows=", neg)

if len(val_df) > 0:
    assert int((val_df.get("is_origin", False) == False).sum()) == 0, "验证集混入了 extra 数据。"
    assert int((val_df.get("is_augmented", False) == True).sum()) == 0, "验证集混入了增强数据。"

print("\n输出文件:")
print("TRAIN_ALL_CSV:", TRAIN_ALL_CSV)
print("VAL_CSV:", VAL_CSV)
print("MASK_CACHE_INDEX_CSV:", MASK_CACHE_INDEX_CSV)
print("DEGRADED_INDEX_CSV:", DEGRADED_INDEX_CSV)
print("MASK_NPZ_CACHE_DIR:", MASK_NPZ_CACHE_DIR)
print("AUG_IMAGE_DIR:", AUG_IMAGE_DIR)
